## Data Visualization - CA2
## Matthew Riddell - D00245674
## Data Preparation and EDA

### Dataset source:

https://www.kaggle.com/datasets/mexwell/dublinbikes-dcc-dataset



### This code will also be uploaded to my github repo here:

https://github.com/Matthew-Riddell/Data-Visualization-CA2

#### Import Packages

In [28]:
import pandas as pd
import numpy as np
import glob

#### grabbing all files in the data folder

In [29]:
# https://www.w3schools.com/python/ref_module_glob.asp
file_path = "C:/Users/Matty/Documents/College Notes & Assignments/Year 5/Data Visualisation and Insight/CAs/CA2/Data-Visualization-CA2/data/*.csv"
files = glob.glob(file_path)

#### list for all files

In [30]:
df_list = []

#### read each file and add them to the list and then add them to a new dataframe 

In [31]:
for file in files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

data = pd.concat(df_list, ignore_index=True)

#### Size of the dataframe

In [32]:
data.shape

(1950289, 11)

#### Display first 5 rows

In [33]:
print("First 5 rows of the dataset:")
print(data.head())

First 5 rows of the dataset:
   STATION ID                 TIME         LAST UPDATED                NAME  \
0           2  2022-01-01 00:00:04  2021-12-31 23:57:39  BLESSINGTON STREET   
1           3  2022-01-01 00:00:04  2021-12-31 23:49:57       BOLTON STREET   
2           4  2022-01-01 00:00:04  2021-12-31 23:58:39        GREEK STREET   
3           5  2022-01-01 00:00:04  2021-12-31 23:51:48    CHARLEMONT PLACE   
4           6  2022-01-01 00:00:04  2021-12-31 23:59:13  CHRISTCHURCH PLACE   

   BIKE_STANDS  AVAILABLE_BIKE_STANDS  AVAILABLE_BIKES STATUS  \
0           20                     10               10   OPEN   
1           20                     19                1   OPEN   
2           20                      9               11   OPEN   
3           40                     17               23   OPEN   
4           20                     13                7   OPEN   

              ADDRESS  LATITUDE  LONGITUDE  
0  Blessington Street   53.3568   -6.26814  
1       Bolton 

#### Display column names

In [34]:
print("Original column names:")
print(data.columns)

Original column names:
Index(['STATION ID', 'TIME', 'LAST UPDATED', 'NAME', 'BIKE_STANDS',
       'AVAILABLE_BIKE_STANDS', 'AVAILABLE_BIKES', 'STATUS', 'ADDRESS',
       'LATITUDE', 'LONGITUDE'],
      dtype='object')


#### Check basic information about the dataset

In [35]:
print("Information about the dataset:")
print(data.info())

Information about the dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1950289 entries, 0 to 1950288
Data columns (total 11 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   STATION ID             int64  
 1   TIME                   object 
 2   LAST UPDATED           object 
 3   NAME                   object 
 4   BIKE_STANDS            int64  
 5   AVAILABLE_BIKE_STANDS  int64  
 6   AVAILABLE_BIKES        int64  
 7   STATUS                 object 
 8   ADDRESS                object 
 9   LATITUDE               float64
 10  LONGITUDE              float64
dtypes: float64(2), int64(4), object(5)
memory usage: 163.7+ MB
None


#### Convert time field into a datetime object

In [36]:
data["TIME"] = pd.to_datetime(data["TIME"])

### Creating new features

#### Create Time features

In [37]:
data["HOUR"] = data["TIME"].dt.hour
data["DAY"] = data["TIME"].dt.day
data["MONTH"] = data["TIME"].dt.month
data["WEEKDAY"] = data["TIME"].dt.day_name()
data["DATE"] = data["TIME"].dt.date

#### Bike usage metric

In [38]:
data["USAGE"] = data["BIKE_STANDS"] - data["AVAILABLE_BIKES"]

#### Weekend Metric

In [ ]:
data["IS_WEEKEND"] = data["WEEKDAY"].isin(["Saturday", "Sunday"])

KeyError: 'weekday'

### Data Cleaning

#### Check for missing values in the dataset

In [ ]:
print("Number of missing values in each column:")
print(data.isna().sum())

In [ ]:
total_missing = data.isna().sum().sum()
print("Total missing values in the dataset:", total_missing)

In [ ]:
nonsense_values = ["?", "error", "missing", "Missing", "NaN", "nan", "N/A", "n/a", "--", " "]

In [ ]:
print("Checking for nonsense values:")
for col in data.columns:
    count = data[col].isin(nonsense_values).sum() 
    if count > 0:
        print(col, "has", count, "nonsense values")

In [ ]:
data = data.replace(nonsense_values, np.nan)

In [ ]:
print("Number of missing values after cleaning nonsense values:")
print(data.isna().sum())

In [ ]:
numeric_columns = ['BIKE_STANDS', 'AVAILABLE_BIKE_STANDS', 'AVAILABLE_BIKES', 'USAGE']

In [ ]:
for col in numeric_columns:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')

print("Shape before dropping rows with missing values:", data.shape)
data = data.dropna()
print("Shape after dropping rows with missing values:", data.shape)

print("Missing values after filling numeric columns with mean:")
print(data.isna().sum())

In [ ]:
print("Detecting outliers using IQR method:\n")

for col in numeric_columns:
    if col in data.columns:
        Q1 = data[col].quantile(0.25) # https://www.w3schools.com/Python/pandas/ref_df_quantile.asp
        Q3 = data[col].quantile(0.75) # https://www.w3schools.com/Python/pandas/ref_df_quantile.asp
        IQR = Q3 - Q1
        lower_limit = Q1 - 1.5 * IQR
        upper_limit = Q3 + 1.5 * IQR
        
        outliers = data[(data[col] < lower_limit) | (data[col] > upper_limit)]
        print(col, "has", outliers.shape[0], "outliers")

In [ ]:
data.head(5)

### Dataset Export

#### Save combined dataset for now

In [ ]:
### data.to_csv("dublinbikes_2022_combined.csv", index=False)